# Value at Risk (VaR) - Single Asset: Historical vs. Variance-Covariance (Parametric)

## 1. Historical Simulation Method (Non-Parametric)

**Theory:** Uses historical returns to estimate potential losses without assuming a specific probability distribution. VaR is calculated directly from empirical percentiles.

**Formulas:**

**Value at Risk (VaR):**

$$
\operatorname{VaR}_{\alpha}^{\text{Hist}}
=
-\widehat{Q}_{1-\alpha}(R_1,\ldots,R_n)
$$

**Expected Shortfall (ES):**

$$
\operatorname{ES}_{\alpha}^{\text{Hist}}
=
-\mathbb{E}\left[
R\mid R\leq\widehat{Q}_{1-\alpha}(R)
\right]
$$

## 2. Variance-Covariance (Parametric) Method

**Theory:** Assumes returns follow a normal distribution. VaR and ES are calculated using the mean, standard deviation, and the appropriate standard normal quantile.

**Formulas:**

**Value at Risk (VaR):**

$$
\operatorname{VaR}_{\alpha}^{\text{Param}}
=
z_{\alpha}\sigma-\mu
$$

**Expected Shortfall (ES):**

$$
\operatorname{ES}_{\alpha}^{\text{Param}}
=
-\mu+\frac{\sigma\phi(z_{\alpha})}{1-\alpha}
$$

**Where:**

- $\alpha$ = Confidence level (e.g., 95% or 99%).
- $\mu$ = Mean return.
- $\sigma$ = Standard deviation of returns.
- $z_{\alpha}=\Phi^{-1}(\alpha)$ = Standard normal quantile (e.g., $z_{0.95}\approx1.645$).
- $\phi(z_{\alpha})$ = Standard normal probability density function (PDF) evaluated at $z_{\alpha}$.
- $\widehat{Q}_{1-\alpha}$ = Empirical lower-tail quantile of historical returns.

**Note:** VaR and ES are expressed as positive losses. The Historical Simulation method uses observed returns, whereas the Variance-Covariance method assumes normally distributed returns.

In [4]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Download real-world data from Yahoo Finance
# We will use Apple Inc. (AAPL) as our example stock
ticker = "AAPL"
start_date = "2023-01-01"
end_date = "2026-01-01"

print(f"Downloading historical data for {ticker}...")
data = yf.download(ticker, period= "3y", auto_adjust=True)

# 2. Calculate daily percentage returns
# We use Adjusted Close to account for dividends and stock splits
returns = data["Close"].pct_change().dropna()

# Convert returns series to a flat 1D array to avoid any alignment issues
returns_arr = returns.to_numpy().flatten()

# 3. Define risk parameters
confidence_level = 0.95  # 95% confidence
holding_period = 1       # 1-day horizon


# METHOD 1: Historical Simulation (Non-parametric)

# Historical VaR is simply the empirical percentile of past returns
# For a 95% confidence level, we look at the worst 5% of outcomes (5th percentile)
var_historical = -np.percentile(returns_arr, (1 - confidence_level) * 100)

# Expected Shortfall is the average of all losses that exceed the VaR threshold
tail_losses = returns_arr[returns_arr <= -var_historical]
es_historical = -np.mean(tail_losses)


# METHOD 2: Variance-Covariance (Parametric)

# Assumes returns follow a normal distribution
from scipy.stats import norm

mean_return = np.mean(returns_arr)
std_dev = np.std(returns_arr)

# Z-score corresponding to the confidence level
z_score = norm.ppf(confidence_level)

# Parametric VaR formula: -(Mean - Z * StdDev)
var_parametric = -(mean_return - z_score * std_dev)

# Parametric ES formula using the standard normal distribution PDF
alpha = 1 - confidence_level
es_parametric = -(
    mean_return - std_dev * (norm.pdf(norm.ppf(alpha)) / alpha)
)


# Print Results

print(f"\n--- Risk Metrics for {ticker} (95% Confidence, 1-Day Horizon) ---")
print(f"Historical VaR:       {var_historical * 100:.2f}%")
print(f"Historical ES:        {es_historical * 100:.2f}%")
print("-" * 50)
print(f"Parametric VaR:       {var_parametric * 100:.2f}%")
print(f"Parametric ES:        {es_parametric * 100:.2f}%")


[*********************100%***********************]  1 of 1 completed


--- Risk Metrics for AAPL (95% Confidence, 1-Day Horizon) ---
Historical VaR:       2.52%
Historical ES:        3.82%
--------------------------------------------------
Parametric VaR:       2.66%
Parametric ES:        3.37%


In [5]:
data

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2023-10-27,165.975189,166.705320,164.603739,164.682673,58499100
2023-10-30,168.017563,168.885825,166.616514,166.764521,51131000
2023-10-31,168.491150,168.619405,165.659439,167.090101,44846000
2023-11-01,171.648422,171.904947,167.849793,168.718055,56934900
2023-11-02,175.200409,175.407598,173.118565,173.177762,77334800
...,...,...,...,...,...
2026-10-21,343.989990,345.250000,338.829987,339.359985,35614100
2026-10-22,348.220001,348.359985,342.279999,344.940002,46898900
